# PersonaPlex — RunPod RTX 5090 Deployment Notebook (with RAG + Tool Calling)

This notebook deploys **[PersonaPlex](https://github.com/NVIDIA/personaplex)** — NVIDIA's real-time, full-duplex
speech-to-speech model with persona/voice control (built on [Moshi](https://arxiv.org/abs/2410.00037)) — on a
**fresh RunPod pod with an RTX 5090 GPU**, including the **RAG (retrieval-augmented generation) and tool-calling**
layer added under `moshi/moshi/rag/` (see `RAG_TOOLCALLING_BUILD_PROMPT.md` at the repo root for the design).

It is derived directly from the repository's own `README.md`, `client/README.md`, `moshi/pyproject.toml`,
`moshi/requirements.txt`, and the server/offline entrypoints (`moshi/moshi/server.py`, `moshi/moshi/offline.py`,
`moshi/moshi/utils/connection.py`, `moshi/moshi/models/loaders.py`, `moshi/moshi/rag/`). No steps are invented —
every command below maps to something the repo actually does or documents.

## What this notebook does

1. Verifies the RunPod environment, GPU and CUDA.
2. Sets up persistent storage on the RunPod volume.
3. Installs system + Python dependencies (including the **Blackwell/RTX 5090‑specific PyTorch build**, and the
   RAG/tool-calling stack: `sentence-transformers`, `transformers`, `faster-whisper`, `pypdf`).
4. Clones/uses the repository (skipped automatically if you already uploaded it — see the important note in
   Section 4 about why RAG requires your own uploaded copy, not the vanilla upstream clone).
5. Configures Hugging Face authentication and downloads the model weights, tokenizer, voices and web UI assets.
6. **Builds the RAG index from `text.txt`** using `python -m moshi.rag.index_builder`.
7. Starts the PersonaPlex backend server (which also serves the prebuilt web UI — no separate frontend build is
   required for normal use) with RAG + tool calling enabled.
8. Verifies the deployment with an automated offline inference smoke test and an HTTP check.
9. Documents GPU optimization notes and troubleshooting steps, including RAG-specific ones.

## What PersonaPlex does *not* have (so these checklist items are intentionally empty)

- **No database** of any kind — the RAG index is a flat `manifest.json` + `chunks.npz` on disk, nothing to
  initialize as a service.
- **No separate "API server"** — the single aiohttp server in `moshi/moshi/server.py` exposes both the WebSocket
  endpoint (`/api/chat`) and the static web UI on one port; RAG/tool-calling runs in-process alongside it.
- **No Gradio/Streamlit app** — the only Gradio dependency is `gradio.networking.setup_tunnel`, an *optional*
  public-URL tunnel (`--gradio-tunnel`), not a Gradio UI. RunPod's own HTTP port proxy makes this unnecessary here.
- **No required "configuration file" to generate** — runtime behavior is controlled entirely by CLI flags
  (including the new `--rag-*` / `--web-search-*` / `--compressor-*` flags) and the HF-hosted `config.json` that
  ships with the model weights.

## Things this notebook *cannot* automate (you must do these yourself)

1. **Accept the NVIDIA Open Model License** for the gated model repo
   [`nvidia/personaplex-7b-v1`](https://huggingface.co/nvidia/personaplex-7b-v1) — log into Hugging Face in a
   browser and click "Agree and access repository". No script can click this for you.
2. **Create a Hugging Face access token** for that same account and have it ready to paste into the auth cell
   below.
3. **Expose the server port (default `8998`) as an HTTP Service** from the RunPod pod's *Connect* page (RunPod
   console action) so the proxy URL works from outside the pod.
4. **Grant microphone permission** in your browser when you open the live web UI — this is a per-user browser
   security prompt.
5. **Get a web-search API key** (Tavily/Serper/Bing) if you want to enable the optional web-search tool — it is
   off by default.

Run the cells **top to bottom**. The only cell you must intentionally run out of the normal flow is the final
**"Stop the server"** cleanup cell — leave it for whenever you actually want to shut the server down.

## 1. Environment sanity checks

Confirms this is a Linux pod with a GPU attached and a supported Python version (`moshi-personaplex` requires
Python ≥ 3.10, per `moshi/pyproject.toml`).

In [ ]:
import platform
import sys

print("Platform:", platform.platform())
print("Python:", sys.version)

assert sys.version_info >= (3, 10), (
    f"PersonaPlex (moshi/pyproject.toml) requires Python >= 3.10, found {sys.version_info}."
)
print("Python version OK.")

In [ ]:
# Confirm the NVIDIA driver sees a GPU at the OS level before we install anything.
!nvidia-smi

## 2. Persistent storage setup (RunPod volume)

RunPod mounts your persistent Network Volume at **`/workspace`**. Anything written there survives pod
stop/start (model weights are multiple GB, so re-downloading them on every restart would be wasteful).
If `/workspace` isn't present (e.g. you're running this notebook somewhere else), we fall back to the home
directory so the notebook still works end-to-end.

We also point Hugging Face's cache (`HF_HOME`) at the persistent volume so `huggingface_hub` downloads
(triggered both by this notebook and internally by `moshi/moshi/server.py` / `moshi/moshi/offline.py`) are
cached once and reused across restarts.

In [ ]:
import os

WORKSPACE = "/workspace" if os.path.isdir("/workspace") else os.path.expanduser("~")
REPO_URL = "https://github.com/MoshiHead/personaplex-original-code-streaming-s-system-try.git"
REPO_DIR = os.path.join(WORKSPACE, "personaplex")
HF_CACHE_DIR = os.path.join(WORKSPACE, ".cache", "huggingface")
HF_REPO_ID = "nvidia/personaplex-7b-v1"   # loaders.DEFAULT_REPO in moshi/moshi/models/loaders.py

SERVER_HOST = "0.0.0.0"   # must be 0.0.0.0 (not "localhost") so RunPod's proxy can reach the server
SERVER_PORT = 8998        # default port used by moshi.server

# RunPod's HTTP port proxy terminates TLS at its edge and forwards plain HTTP to the container, so by
# default we do NOT enable the app's own self-signed TLS (--ssl). Flip this to True only if you plan to
# expose SERVER_PORT directly as a raw TCP port instead of through RunPod's HTTP proxy.
USE_APP_TLS = False

# An RTX 5090 has 32GB of VRAM, comfortably enough for this 7B model in bf16 -- CPU offload should not be
# needed. Flip to True only if you hit CUDA OOM (requires the `accelerate` package, installed below anyway).
USE_CPU_OFFLOAD = False

# --- RAG (retrieval-augmented generation) + tool calling, see moshi/moshi/rag/ ---------------------------
RAG_ENABLED = True
RAG_SOURCE_FILE = os.path.join(REPO_DIR, "text.txt")     # knowledge base to build the index from
RAG_INDEX_DIR = os.path.join(WORKSPACE, "rag_index")      # persisted on the volume, rebuilt in Section 9b
RAG_TOP_K = 3
RAG_MIN_SCORE = 0.35
RAG_TRIGGER_SCORE = 0.45
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

# Context compressor: a small separate instruct LLM that turns retrieved passages into a short spoken
# grounding note. Runs on its own thread/executor; an RTX 5090 has plenty of VRAM headroom for both this
# and the 7B PersonaPlex model, so 4-bit quantization is off by default (flip to True to save VRAM).
COMPRESSOR_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
COMPRESSOR_DEVICE = "cuda"
COMPRESSOR_4BIT = False

# STT (user-speech transcription) used to decide *what* to retrieve for -- kept on CPU by default so it
# never contends with the CUDA-graphed PersonaPlex model for GPU execution.
STT_MODEL_NAME = "base"
STT_DEVICE = "cpu"

# Web search fallback tool: off by default (no API key configured). Set WEB_SEARCH_ENABLED = True and
# fill in WEB_SEARCH_API_KEY (Tavily/Serper/Bing) to let the router fall back to live web search when
# local retrieval confidence is low.
WEB_SEARCH_ENABLED = False
WEB_SEARCH_PROVIDER = "tavily"   # tavily | serper | bing
WEB_SEARCH_API_KEY = os.environ.get("WEB_SEARCH_API_KEY", "")

os.makedirs(WORKSPACE, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
# Keep PATH aware of ~/.local/bin, where moshi/moshi/utils/connection.py installs `mkcert` if --ssl is used.
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ.get("PATH", "")

print("WORKSPACE     :", WORKSPACE)
print("REPO_DIR      :", REPO_DIR)
print("HF_HOME       :", os.environ["HF_HOME"])
print("USE_APP_TLS   :", USE_APP_TLS)
print("RAG_ENABLED   :", RAG_ENABLED)
print("RAG_INDEX_DIR :", RAG_INDEX_DIR)

## 3. System package installation

Per the repo's `README.md` prerequisites: install the **Opus codec development library** before installing the
Python package (it's required by `sphn`, which PersonaPlex uses for Opus audio streaming over the WebSocket).
We also make sure `git` is present for cloning.

In [ ]:
import os

SUDO = "" if os.geteuid() == 0 else "sudo "

!{SUDO}apt-get update -qq
!{SUDO}apt-get install -y -qq --no-install-recommends git ca-certificates libopus-dev
print("System packages installed.")

## 4. Repository cloning

If `REPO_DIR` doesn't already contain the project, it is cloned fresh from GitHub. If it's already there
(e.g. you uploaded your own copy of this repo to the volume beforehand), cloning is skipped automatically —
this cell is safe to re-run on every pod restart.

**Important for RAG/tool calling:** the RAG layer (`moshi/moshi/rag/`), the RunPod-facing `--rag-*`/`--web-search-*`/
`--compressor-*` server flags, and `text.txt` (the default knowledge base) are additions made **on top of** the
upstream `NVIDIA/personaplex` repo — they are not part of the public GitHub clone at `REPO_URL`. To use RAG, upload
your modified working copy (this notebook's own repo) to `REPO_DIR` on the RunPod volume **before** running this
cell, so the clone is skipped and your `moshi/moshi/rag/` package + `text.txt` are what's actually used below. If
this cell ends up cloning fresh from `REPO_URL` instead, the server will still start, but RAG/tool calling will
report itself disabled (no index directory) until you supply your own `--rag-index-dir`.

In [ ]:
import pathlib
import subprocess

repo_marker = pathlib.Path(REPO_DIR) / "moshi" / "pyproject.toml"

if repo_marker.exists():
    print(f"Repository already present at {REPO_DIR}, skipping clone.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print(f"Cloned into {REPO_DIR}.")

assert repo_marker.exists(), f"Expected {repo_marker} to exist after cloning/upload."


## 5. Python dependency installation

The repo's documented install command is:

```bash
pip install moshi/.
```

which installs from `moshi/pyproject.toml` (`numpy`, `safetensors`, `huggingface-hub`, `einops`,
`sentencepiece`, `sounddevice`, `sphn`, `torch>=2.2,<2.5`, `aiohttp`, and — for RAG/tool calling —
`sentence-transformers`, `transformers`, `accelerate`, `faster-whisper`, `pypdf`).

### RTX 5090 / Blackwell note

The pinned `torch<2.5` build does **not** ship CUDA kernels for Blackwell (RTX 50‑series, `sm_120`) GPUs. The
repo's `README.md` explicitly documents the fix for this
([NVIDIA/personaplex#2](https://github.com/NVIDIA/personaplex/issues/2)): reinstall PyTorch from the `cu130`
wheel index *after* the base install. This intentionally overrides the `<2.5` pin — that's expected and is the
upstream-recommended fix, not a mistake.

In [ ]:
%pip install -q --upgrade pip setuptools wheel
%pip install -q "{REPO_DIR}/moshi/." 

In [ ]:
# Blackwell (RTX 5090) requires CUDA-13.0-built PyTorch wheels. This intentionally supersedes the
# torch<2.5 pin from moshi/pyproject.toml -- see README.md "Extra step for Blackwell based GPUs".
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130

In [ ]:
# `accelerate` enables --cpu-offload (README's "CPU Offload" section); `gradio` enables the optional
# --gradio-tunnel fallback for public exposure if you ever need it instead of RunPod's HTTP proxy.
%pip install -q accelerate gradio

## 6. CUDA / GPU verification

Confirms PyTorch can see the RTX 5090 and that the installed build actually has working CUDA kernels for it
(a bf16 matmul smoke test) — this is the check that would have failed before the `cu130` reinstall above if it
had been skipped.

In [ ]:
import torch

print("Torch version      :", torch.__version__)
print("Torch CUDA version :", torch.version.cuda)
print("CUDA available     :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected by PyTorch. Confirm this RunPod pod has an RTX 5090 GPU attached "
        "and that the driver is healthy (see the `nvidia-smi` output above)."
    )

device_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print("GPU                :", device_name)
print("Compute capability :", capability)

if "5090" not in device_name and "RTX 50" not in device_name:
    print(f"WARNING: expected an RTX 5090, found '{device_name}'. Continuing anyway.")

# Smoke test: this is exactly the kind of op that fails with
# "no kernel image is available for execution on the device" on Blackwell if torch wasn't
# reinstalled from the cu130 index.
x = torch.randn(4096, 4096, device="cuda", dtype=torch.bfloat16)
y = x @ x
torch.cuda.synchronize()
print("bf16 CUDA matmul smoke test OK, result shape:", tuple(y.shape))

## 7. Hugging Face authentication

**Manual step required before running this cell:** log into Hugging Face in a browser, open
[`nvidia/personaplex-7b-v1`](https://huggingface.co/nvidia/personaplex-7b-v1), and click **"Agree and access
repository"** to accept the NVIDIA Open Model License. Then create an access token (read access is enough) at
<https://huggingface.co/settings/tokens>.

The repo's `README.md` documents this as `export HF_TOKEN=<YOUR_HUGGINGFACE_TOKEN>`; the cell below does the
same thing from inside the notebook, via a hidden prompt so the token isn't echoed into cell output.

In [ ]:
from getpass import getpass

from huggingface_hub import login

# hf_token = os.environ.get("HF_TOKEN")
# sa token...not mine
hf_token = 'tLNSyNjFduNaLUbvyxosVqiGwuAtiQPOTt'    
if not hf_token:
    hf_token = getpass("Enter your Hugging Face access token (input hidden): ")

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
print("Logged in to Hugging Face Hub.")

## 8. Model downloading

`moshi/moshi/server.py` and `moshi/moshi/offline.py` both lazily call `hf_hub_download` for each asset the
first time they need it. We pre-fetch the same files here so that (a) any license/token problem surfaces now
with a clear error instead of mid-startup, and (b) everything is already warm in the `HF_HOME` cache before we
launch the server.

Assets (from `moshi/moshi/models/loaders.py` and `moshi/moshi/server.py`):
- `config.json` — model config
- `tokenizer_spm_32k_3.model` — SentencePiece text tokenizer
- `tokenizer-e351c8d8-checkpoint125.safetensors` — Mimi codec weights
- `model.safetensors` — Moshi/PersonaPlex LM weights
- `voices.tgz` — packaged voice-prompt embeddings (NATF0‑3, NATM0‑3, VARF0‑4, VARM0‑4)
- `dist.tgz` — the **prebuilt web UI** (this is why no separate frontend build step is needed)

In [ ]:
import tarfile

from huggingface_hub import hf_hub_download

ASSET_FILES = [
    "config.json",
    "tokenizer_spm_32k_3.model",
    "tokenizer-e351c8d8-checkpoint125.safetensors",
    "model.safetensors",
    "voices.tgz",
    "dist.tgz",
]

downloaded = {}
try:
    for fname in ASSET_FILES:
        # No explicit cache_dir: this honors HF_HOME (set in step 2) so the notebook and the server
        # subprocess we launch later (which inherits the same env) share one cache.
        path = hf_hub_download(HF_REPO_ID, fname)
        downloaded[fname] = path
        print(f"OK  {fname} -> {path}")
except Exception as e:
    raise RuntimeError(
        "Failed to download model assets from "
        f"https://huggingface.co/{HF_REPO_ID}. This almost always means either:\n"
        "  1) you have not clicked 'Agree and access repository' on that model page yet, or\n"
        "  2) the HF_TOKEN you supplied doesn't belong to the account that accepted the license, or\n"
        "  3) the token is invalid/expired.\n"
        f"Original error: {e}"
    )

In [ ]:
import pathlib

# Pre-extract the tarballs once, exactly like _get_voice_prompt_dir / _get_static_path do in
# moshi/moshi/server.py, so the first real request doesn't pay the extraction cost.
for tgz_name in ("voices.tgz", "dist.tgz"):
    tgz_path = pathlib.Path(downloaded[tgz_name])
    out_dir = tgz_path.parent / tgz_name.replace(".tgz", "")
    if not out_dir.exists():
        with tarfile.open(tgz_path, "r:gz") as tar:
            tar.extractall(path=tgz_path.parent)
    print(f"{tgz_name} -> {out_dir} ({'already extracted' if out_dir.exists() else 'extracted now'})")

## 9b. Build the RAG index from `text.txt`

`moshi/moshi/rag/index_builder.py` reads `RAG_SOURCE_FILE`, splits it into sentence-bounded chunks (~30 words
each, never breaking a sentence), embeds every chunk with the `EMBEDDING_MODEL_NAME` sentence-transformer, and
tokenizes each chunk with the LM's own SentencePiece tokenizer (so mid-stream `<ref>` injection never has to
re-tokenize retrieved text). It writes `manifest.json` + `chunks.npz` into `RAG_INDEX_DIR`.

This only needs to be re-run when `text.txt` (or another knowledge base you point `RAG_SOURCE_FILE` at) changes
— the index persists on the RunPod volume across pod restarts, same as the model weights. The build is CPU-only
and takes seconds for a document this size.

In [ ]:
import subprocess
import sys

if not RAG_ENABLED:
    print("RAG_ENABLED = False, skipping index build.")
elif not os.path.exists(RAG_SOURCE_FILE):
    raise FileNotFoundError(
        f"RAG_SOURCE_FILE not found: {RAG_SOURCE_FILE}. Either upload text.txt to that path, "
        "point RAG_SOURCE_FILE at a different file/directory, or set RAG_ENABLED = False."
    )
else:
    index_cmd = [
        sys.executable, "-m", "moshi.rag.index_builder",
        "--input", RAG_SOURCE_FILE,
        "--output-dir", RAG_INDEX_DIR,
        "--hf-repo", HF_REPO_ID,
        "--embedding-model-name", EMBEDDING_MODEL_NAME,
        "--embedding-device", "cpu",
    ]
    print("Building RAG index:", " ".join(index_cmd))
    result = subprocess.run(
        index_cmd, cwd=os.path.join(REPO_DIR, "moshi"), env=os.environ.copy(),
        capture_output=True, text=True,
    )
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError("RAG index build failed -- see output above.")

    manifest_path = os.path.join(RAG_INDEX_DIR, "manifest.json")
    assert os.path.exists(manifest_path), f"Expected {manifest_path} to exist after building the index."
    print(f"\nRAG index ready at {RAG_INDEX_DIR}")

## 9. TLS certificate directory (only used if `USE_APP_TLS = True`)

The README's default local-machine workflow is:

```bash
SSL_DIR=$(mktemp -d); python -m moshi.server --ssl "$SSL_DIR"
```

`--ssl` makes `moshi/moshi/utils/connection.py` auto-install `mkcert` and generate a **self-signed**
certificate in that directory. That's appropriate for direct LAN/local access, but on RunPod we're putting the
server behind RunPod's own HTTPS proxy (Section 11), which already terminates TLS at the edge — adding a second,
self-signed TLS layer underneath it would just produce certificate warnings for no benefit. We still create the
directory here so you can flip `USE_APP_TLS = True` above if you'd rather expose `SERVER_PORT` as a raw TCP port
instead of through the HTTP proxy.

In [ ]:
import tempfile

SSL_DIR = tempfile.mkdtemp(prefix="personaplex_ssl_")
print("SSL_DIR:", SSL_DIR, "(only used if USE_APP_TLS = True)")

## 10. Backend (+ web UI + RAG/tool calling) startup

This is the repo's **recommended and only documented launch method** — `python -m moshi.server`. It is a
single aiohttp process that serves:
- the WebSocket endpoint `/api/chat` (the real-time speech protocol),
- the prebuilt web UI from `dist.tgz` at `/` (downloaded in Section 8) — there is **no separate frontend
  process to start** for normal use, and
- the RAG/tool-calling layer in-process (`moshi/moshi/rag/`) — local retrieval against `RAG_INDEX_DIR`, the
  score-gated tool router, the optional web-search tool, and the context compressor.

RAG/tool-calling flags are only passed when `RAG_ENABLED = True` **and** the index was actually built in
Section 9b; otherwise the server falls back to running with `--no-rag` (ordinary conversation, no retrieval).
The server also validates `--rag-min-score <= --rag-trigger-score` at startup and fails loudly if that's
violated, rather than silently misfiring on the first turn.

A notebook cell that calls `web.run_app(...)` directly would block the kernel forever, so we launch it as a
background subprocess and log to a file, then poll that log in the next cell.

In [ ]:
import subprocess
import sys

env = os.environ.copy()
LOG_PATH = os.path.join(WORKSPACE, "personaplex_server.log")

cmd = [sys.executable, "-m", "moshi.server", "--host", SERVER_HOST, "--port", str(SERVER_PORT)]
if USE_APP_TLS:
    cmd += ["--ssl", SSL_DIR]
if USE_CPU_OFFLOAD:
    cmd += ["--cpu-offload"]

index_ready = RAG_ENABLED and os.path.exists(os.path.join(RAG_INDEX_DIR, "manifest.json"))
if index_ready:
    cmd += [
        "--rag-index-dir", RAG_INDEX_DIR,
        "--rag-top-k", str(RAG_TOP_K),
        "--rag-min-score", str(RAG_MIN_SCORE),
        "--rag-trigger-score", str(RAG_TRIGGER_SCORE),
        "--embedding-model-name", EMBEDDING_MODEL_NAME,
        "--embedding-device", "cpu",
        "--compressor-model-name", COMPRESSOR_MODEL_NAME,
        "--compressor-device", COMPRESSOR_DEVICE,
        "--stt-model-name", STT_MODEL_NAME,
        "--stt-device", STT_DEVICE,
    ]
    if COMPRESSOR_4BIT:
        cmd += ["--compressor-4bit"]
    if WEB_SEARCH_ENABLED:
        if not WEB_SEARCH_API_KEY:
            raise RuntimeError(
                "WEB_SEARCH_ENABLED = True but WEB_SEARCH_API_KEY is empty. Set it above (Section 2) "
                "or set WEB_SEARCH_ENABLED = False."
            )
        cmd += [
            "--web-search-enabled",
            "--web-search-provider", WEB_SEARCH_PROVIDER,
            "--web-search-api-key", WEB_SEARCH_API_KEY,
        ]
else:
    cmd += ["--no-rag"]
    print("RAG index not found/enabled -- launching with --no-rag (ordinary conversation only). "
          "Run Section 9b to build the index if you want RAG/tool calling.")

print("Launching:", " ".join(cmd))

log_file = open(LOG_PATH, "w")
server_proc = subprocess.Popen(
    cmd, cwd=os.path.join(REPO_DIR, "moshi"), env=env, stdout=log_file, stderr=subprocess.STDOUT,
)
print(f"Server launched with PID {server_proc.pid}. Logs: {LOG_PATH}")

In [ ]:
import time

def tail(path, n=60):
    with open(path) as f:
        return "".join(f.readlines()[-n:])

READY_MARKER = "Access the Web UI directly at"
TIMEOUT_S = 900  # first run downloads + loads a multi-GB model; be generous
POLL_S = 5

start = time.time()
ready = False
while time.time() - start < TIMEOUT_S:
    if server_proc.poll() is not None:
        print(tail(LOG_PATH))
        raise RuntimeError(
            f"Server process exited early with return code {server_proc.returncode}. See log above."
        )
    if READY_MARKER in open(LOG_PATH).read():
        ready = True
        break
    time.sleep(POLL_S)

print(tail(LOG_PATH))

if not ready:
    raise TimeoutError(
        f"Server did not report readiness within {TIMEOUT_S}s. Check the log tail above -- "
        "the most common causes are a slow first-time model download or a CUDA/driver mismatch."
    )

print("\nServer is up and warmed up.")

## 11. Expose the port & get the access URL

RunPod will not route external traffic to a port unless you explicitly expose it. In the pod's **Connect**
page, add an **HTTP Service** for `SERVER_PORT` (default `8998`) if you haven't already. RunPod then serves it
at `https://<POD_ID>-<PORT>.proxy.runpod.net`, with RunPod terminating TLS — which is exactly why
`USE_APP_TLS = False` is the right default here (see Section 9).

In [ ]:
pod_id = os.environ.get("RUNPOD_POD_ID")

if pod_id:
    public_url = f"https://{pod_id}-{SERVER_PORT}.proxy.runpod.net"
    print("RunPod public URL (requires SERVER_PORT to be exposed as an HTTP Service on the pod's Connect page):")
    print(" ", public_url)
else:
    print(
        "RUNPOD_POD_ID was not found in the environment. If you are on RunPod, expose "
        f"port {SERVER_PORT} as an HTTP Service from the pod's Connect page and use the proxy URL shown there."
    )

scheme = "https" if USE_APP_TLS else "http"
print(f"Local URL inside the pod: {scheme}://localhost:{SERVER_PORT}")

## 12. Verification — offline inference smoke test

This runs the repo's documented `moshi.offline` "Assistant example" exactly as given in `README.md` — it
streams `assets/test/input_assistant.wav` through the model with voice prompt `NATF2.pt` and writes an output
WAV + JSON transcript. This doesn't need a browser, microphone, or open port, so it's a good way to confirm the
backend, weights and GPU are all working correctly before testing the live UI.

In [ ]:
import subprocess
import sys

offline_cmd = [
    sys.executable, "-m", "moshi.offline",
    "--voice-prompt", "NATF2.pt",
    "--input-wav", "assets/test/input_assistant.wav",
    "--seed", "42424242",
    "--output-wav", "output_assistant.wav",
    "--output-text", "output_assistant.json",
]

result = subprocess.run(
    offline_cmd, cwd=REPO_DIR, env=os.environ.copy(), capture_output=True, text=True,
)

print(result.stdout[-4000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
    raise RuntimeError("Offline smoke test failed -- see output above.")

print("\nOffline smoke test succeeded.")

In [ ]:
import json

from IPython.display import Audio, display

output_wav_path = os.path.join(REPO_DIR, "output_assistant.wav")
output_json_path = os.path.join(REPO_DIR, "output_assistant.json")

display(Audio(output_wav_path))

with open(output_json_path) as f:
    tokens = json.load(f)
print("Generated text tokens (joined):")
print("".join(tokens))

## 13. Verification — HTTP check on the live server

Confirms the running server process answers on `/` (the web UI's `index.html`, served from `dist.tgz`).

In [ ]:
import ssl
import urllib.request

scheme = "https" if USE_APP_TLS else "http"
ctx = ssl._create_unverified_context() if USE_APP_TLS else None

with urllib.request.urlopen(f"{scheme}://localhost:{SERVER_PORT}/", context=ctx, timeout=15) as resp:
    print("HTTP status :", resp.status)
    print("Content-Type:", resp.headers.get("Content-Type"))
    assert resp.status == 200, f"Expected 200, got {resp.status}"

print("Web UI is being served correctly.")

## 14. Using the live web UI

Open the URL printed in Section 11 in a browser, allow microphone access when prompted (this is the one
browser permission step that can't be automated), and start talking.

### Voices (from `README.md`)

```
Natural(female): NATF0, NATF1, NATF2, NATF3
Natural(male):   NATM0, NATM1, NATM2, NATM3
Variety(female): VARF0, VARF1, VARF2, VARF3, VARF4
Variety(male):   VARM0, VARM1, VARM2, VARM3, VARM4
```

### Example role prompts (from `README.md`)

- Assistant role: `You are a wise and friendly teacher. Answer questions or provide advice in a clear and
  engaging way.`
- Casual conversation: `You enjoy having a good conversation.`
- Customer service example: `You work for CitySan Services which is a waste management company and your name
  is Ayelen Lucero. Information: Verify customer name Omar Torres. Current schedule: every other week.
  Upcoming pickup: April 12th. Compost bin service available for $8/month add-on.`

See the repo's `README.md` "Prompting Guide" section for more examples and guidance.

### Testing RAG grounding

With the default `text.txt` knowledge base (about the company RobotBulls), ask the assistant something only
that document would know, e.g. *"What GPUs does RobotBulls use for its AI infrastructure?"* or *"What is
Temporal Window Smoothing?"*. If the RAG index is active, you should hear a short natural stalling phrase
(the `<lookup>` injection) followed by a grounded answer. Check `personaplex_server.log` for lines like
`[rag] grounded turn (top_score=0.5xx, web=False): '...'` to confirm retrieval actually fired versus a plain
conversational turn. Ask something the document doesn't cover to see the plain (ungrounded) path instead.

## 15. GPU optimization notes for RTX 5090

- **bf16 by default**: `moshi/moshi/models/loaders.py:get_moshi_lm` loads the LM in `torch.bfloat16`. Blackwell
  has fast native bf16 Tensor Core throughput, so no dtype changes are needed.
- **`--cpu-offload` is unnecessary here**: it exists in `server.py`/`offline.py` for GPUs with insufficient
  VRAM (via `accelerate`'s `infer_auto_device_map`). An RTX 5090's 32GB comfortably fits this 7B model plus the
  Mimi codec; only enable `USE_CPU_OFFLOAD` if you're also running other large workloads on the same GPU.
- **One process = one GPU's worth of model**: `ServerState.__init__` loads two Mimi instances and the LM once
  per process and keeps them resident (`streaming_forever`). Don't launch a second `moshi.server` process
  against the same GPU unless you've confirmed there's VRAM headroom — check with `nvidia-smi`.
- **Warmup cost is already handled**: `state.warmup()` runs 4 dummy frames through the full encode → LM →
  decode path right after model load, ahead of any real connections — that's the per-process latency you saw
  the log wait for in Section 10, not something to optimize further.
- **CUDA build must match the GPU**: Blackwell (`sm_120`) needs the `cu130` PyTorch wheels installed in
  Section 5. If you ever see `no kernel image is available for execution on the device`, that cell needs to be
  re-run (something likely reinstalled a different torch build afterward).
- **RAG/tool-calling's extra VRAM cost**: with `COMPRESSOR_DEVICE = "cuda"` and `COMPRESSOR_4BIT = False`, the
  ~1.5B-parameter compressor model adds a few GB of VRAM in fp16 alongside the 7B PersonaPlex model and Mimi —
  still comfortable on a 32GB RTX 5090. If you're VRAM-constrained (e.g. also running other workloads), set
  `COMPRESSOR_4BIT = True` (requires `bitsandbytes`, `pip install moshi/.[rag-4bit]`) or `COMPRESSOR_DEVICE =
  "cpu"`. The embedding model (`all-MiniLM-L6-v2`) and STT model (`faster-whisper` "base") default to CPU and
  add negligible GPU load either way.
- **Single-worker GPU-serialized execution still holds for RAG**: `moshi/moshi/rag/session.py`'s
  `RagSession.pre_step_hook()` only ever calls into `lm_gen.step()` (via `inject_text_tokens`) from the same
  `opus_loop` coroutine that already owns every other `lm_gen.step()` call — STT, retrieval, web search and
  compression run in background tasks/executors that never touch `lm_gen`/`mimi` directly, so CUDA graph
  capture/replay for the main model is never interleaved with concurrent calls.

## 16. Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `401`/`403` downloading model assets | License not accepted, or `HF_TOKEN` doesn't belong to the account that accepted it | Re-check Section 7: accept the license at the model page with the **same** account whose token you pasted |
| `no kernel image is available for execution on the device` | Torch build doesn't have Blackwell (`sm_120`) kernels | Re-run the `cu130` reinstall cell in Section 5, then re-run Section 6's smoke test |
| `ImportError` / build errors mentioning `opus` while installing `sphn` | `libopus-dev` missing | Re-run Section 3, then re-run Section 5 |
| Server process exits immediately, log shows a CUDA OOM | Not enough free VRAM (e.g. another process holds the GPU) | Check `nvidia-smi`; consider `USE_CPU_OFFLOAD = True` (Section 2) and re-run Sections 10‑11 |
| Browser blocks microphone / `getUserMedia` fails | Page wasn't loaded over a secure context | Use the RunPod **proxy** URL from Section 11 (HTTPS at the edge), not a plain `http://<pod-ip>:8998` URL |
| Can't reach the URL from outside the pod at all | Port not exposed | In the RunPod console, add `SERVER_PORT` as an HTTP Service on the pod's Connect page |
| First launch seems to hang for several minutes | Normal — first run downloads multi-GB weights and extracts `voices.tgz`/`dist.tgz` | Watch the log tail printed by Section 10; increase `TIMEOUT_S` if your network is slow |
| `mkcert` warnings in the log | Only relevant when `USE_APP_TLS = True`; `moshi/moshi/utils/connection.py` falls back to plain HTTP automatically if `mkcert` can't be installed | Safe to ignore in default (proxy) mode |
| Server logs `RAG/tool-calling enabled = False` at startup | `RAG_ENABLED = False`, or Section 9b never ran / index build failed / `--rag-index-dir` wasn't found | Re-run Section 9b, confirm `manifest.json` exists under `RAG_INDEX_DIR`, then re-run Section 10 |
| `RagConfigError: rag_min_score (...) must be <= rag_trigger_score (...)` at startup | `RAG_MIN_SCORE > RAG_TRIGGER_SCORE` in Section 2 | Fix the two values in Section 2 so `RAG_MIN_SCORE <= RAG_TRIGGER_SCORE`, then re-run Section 10 |
| Log shows `Failed to load RAG index ...` / `Failed to load STT model ...` | Missing/corrupt index, or `faster-whisper` couldn't download its model (no internet egress, or `HF_HOME` issue) | RAG disables itself gracefully (server still runs); check the warning text, fix the underlying cause, and restart the server |
| RAG never seems to fire (`[rag] plain turn` for everything) | `RAG_MIN_SCORE`/`RAG_TRIGGER_SCORE` too strict for this embedding model, or the question genuinely isn't covered by `text.txt` | Lower `RAG_TRIGGER_SCORE` in Section 2 and re-run Section 10; check `top_score` values logged per turn to calibrate |
| `WEB_SEARCH_ENABLED = True but WEB_SEARCH_API_KEY is empty` | Forgot to set the API key | Set `WEB_SEARCH_API_KEY` (Section 2) or set `WEB_SEARCH_ENABLED = False` |
| Compressor load warning, RAG still runs with shorter/plainer grounding notes | `COMPRESSOR_MODEL_NAME` failed to download/load (network, VRAM) | This is a graceful degrade (extractive fallback), not a crash; fix the underlying issue and restart if you want the LLM compressor back |

### Recap: what genuinely cannot be automated by this notebook
1. Clicking "Agree and access repository" on the gated HF model page.
2. Issuing the HF access token for that account.
3. Exposing `SERVER_PORT` as an HTTP Service in the RunPod console.
4. The browser's microphone-permission prompt.
5. Obtaining a web-search API key (Tavily/Serper/Bing) if you want the optional web-search tool.

## 17. Stop the server (run only when you want to shut it down)

This is a management utility cell, **not** part of the linear startup flow — running it will terminate the
backend you just verified above. Run it deliberately when you're done with the session, not as part of a
top-to-bottom "Run All".

In [ ]:
# Intentionally NOT meant to run automatically as part of the startup sequence above.
try:
    server_proc.terminate()
    server_proc.wait(timeout=15)
    print(f"Server process {server_proc.pid} stopped.")
except NameError:
    print("No server_proc in scope -- nothing to stop.")
except subprocess.TimeoutExpired:
    server_proc.kill()
    print(f"Server process {server_proc.pid} killed after not stopping gracefully.")